In [1]:
from pyspark.sql import functions as F

# Bronze-Daten aus dem Lakehouse lesen
df_bronze = spark.table("bronze.vehicle_diagnostics")

# Anzahl Datensätze vor der Bereinigung
print("Bronze-Datensätze:", df_bronze.count())

# Datentypen prüfen
df_bronze.printSchema()

# Rohdaten anzeigen
display(df_bronze)



StatementMeta(, c67f68ea-ad08-4b22-bc03-a354309a7a18, 3, Finished, Available, Finished, False)

Bronze-Datensätze: 5
root
 |-- TestID: long (nullable = true)
 |-- ECU: string (nullable = true)
 |-- DTC: string (nullable = true)
 |-- TestResult: string (nullable = true)
 |-- TestTimestamp: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 3caad8a3-ae2f-4065-8fdb-8442ca6e145b)

In [2]:

from pyspark.sql import functions as F

# 1. Bronze-Daten lesen
df_bronze = spark.table("bronze.vehicle_diagnostics")

# 2. Duplikate anhand der TestID entfernen
df_silver = df_bronze.dropDuplicates(["TestID"])

# 3. Ungültige und fehlende Werte entfernen
df_silver = df_silver.filter(
    F.col("TestID").isNotNull() &
    F.col("ECU").isNotNull() &
    F.col("TestResult").isNotNull() &
    F.col("TestTimestamp").isNotNull()
)

# 4. Teststatus vereinheitlichen
df_silver = df_silver.withColumn(
    "TestResult",
    F.upper(F.trim(F.col("TestResult")))
)

# 5. DTC-Code normalisieren
# Beispiel: 0xP0300 -> P0300
df_silver = df_silver.withColumn(
    "DTC",
    F.upper(
        F.regexp_replace(F.trim(F.col("DTC")), r"(?i)^0x", "")
    )
)

# 6. Zeitstempel konvertieren
df_silver = df_silver.withColumn(
    "TestTimestamp",
    F.to_timestamp(
        F.col("TestTimestamp"),
        "yyyy-MM-dd HH:mm:ss"
    )
)

# 7. Nur gültige Teststatus und Zeitstempel zulassen
df_silver = df_silver.filter(
    F.col("TestResult").isin("PASS", "FAIL") &
    F.col("TestTimestamp").isNotNull()
)

# 8. DTC-Format prüfen
df_silver = df_silver.withColumn(
    "DTC_Valid",
    F.col("DTC").rlike("^[PCBU][0-9A-F]{4}$")
)

# 9. Ergebnis anzeigen
display(df_silver)

print("Silver-Datensätze:", df_silver.count())


StatementMeta(, c67f68ea-ad08-4b22-bc03-a354309a7a18, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 010325f5-91ea-45ce-ab62-7b5605383c27)

Silver-Datensätze: 5


In [3]:
# Silver Layer – bereinigte Daten speichern

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.vehicle_diagnostics_clean")

print("Silver-Tabelle erfolgreich gespeichert!")


StatementMeta(, c67f68ea-ad08-4b22-bc03-a354309a7a18, 5, Finished, Available, Finished, False)

Silver-Tabelle erfolgreich gespeichert!


In [4]:
# Silver-Tabelle aus dem Lakehouse lesen
df_check = spark.table("silver.vehicle_diagnostics_clean")

# Ergebnis kontrollieren
display(df_check)

print("Anzahl Silver-Datensätze:", df_check.count())

df_check.printSchema()


StatementMeta(, c67f68ea-ad08-4b22-bc03-a354309a7a18, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 11a83635-2a5a-4750-9134-aec238125e9f)

Anzahl Silver-Datensätze: 5
root
 |-- TestID: long (nullable = true)
 |-- ECU: string (nullable = true)
 |-- DTC: string (nullable = true)
 |-- TestResult: string (nullable = true)
 |-- TestTimestamp: timestamp (nullable = true)
 |-- DTC_Valid: boolean (nullable = true)

